In [ ]:
import scipy
import time
import numpy as np

import matplotlib.pyplot as plt
from adaptive_latents import datasets
rng = np.random.default_rng(0)
from sklearn.decomposition import PCA

In [ ]:
from adaptive_latents.estimator import TypicalEstimator
from adaptive_latents.timed_data_source import ArrayWithTime
from adaptive_latents.prosvd import RandomProjection

class RandomizedFilterer(TypicalEstimator):
    # https://doi.org/10.48550/arXiv.2601.08685
    def __init__(self, *, k=1, rng, input_streams=None, output_streams=None, log_level=None, on_nan_width=None):
        super().__init__(input_streams=input_streams, output_streams=output_streams, log_level=log_level, on_nan_width=on_nan_width)
        self.rng: np.random.Generator = rng
        self.k = k

        self.sign_vector = None
        self.projection = None

    def pre_initialization_fit_for_X(self, X):
        self.sign_vector = np.ones(X.shape[1])
        self.sign_vector[self.rng.choice(a=X.shape[1], size=X.shape[1]//2, replace=False)] = -1

        self.projection = np.zeros((X.shape[1], self.k))
        indices_to_keep = self.rng.choice(a=X.shape[1], size=self.k, replace=False)
        self.projection[indices_to_keep, np.arange(self.k)] = 1

        self.is_initialized = True

    def partial_fit_for_X(self, X):
        pass

    def transform_for_X(self, X):
        out = []
        for row in X:
            out.append(self.projection.T @ scipy.fft.fft(self.sign_vector * row))
        return ArrayWithTime.from_transformed_data(out, X)

    def instance_get_params(self, deep=True):
        return {}


In [ ]:
a = datasets.Odoherty21Dataset().neural_data

embeddings = dict()

embeddings['random_filtered'] = RandomizedFilterer(k=2,rng=rng).offline_run_on(a)
embeddings['proj_ortho'] = (rp:=RandomProjection(k=4, mode='orthonormal')).offline_run_on(a)
embeddings['pca'] = PCA(n_components=4).fit_transform(a)
embeddings['pca'] = ArrayWithTime(embeddings['pca'], t=a.t)
# embeddings['proj_gauss'] = RandomProjection(k=4, mode='gaussian').offline_run_on(a)


M, N = a.shape
P = np.eye(N)[rng.choice(N, size=2, replace=False)] @ scipy.linalg.dft(N, scale=None) @ np.diag(rng.integers(0,2, size=N) * 2 - 1)
P = P / np.linalg.norm(P,axis=1)[:,None]
embeddings['rf_manual'] = a @ P.T



In [ ]:
embeddings['original'], _ = ArrayWithTime.align_indices(a, embeddings['random_filtered'])
embeddings['pca'], _ = ArrayWithTime.align_indices(embeddings['pca'], embeddings['random_filtered'])
for embedding in embeddings.values():
    assert (embedding.t == embeddings['original'].t).all()

In [ ]:
distances = dict()
for key, embedding in embeddings.items():
    if np.isreal(embedding).all():
        distances[key] = scipy.spatial.distance.pdist(embedding)
    else:
        distances[key] = scipy.spatial.distance.squareform(scipy.spatial.distance_matrix(embedding, embedding))

In [ ]:
s = rng.choice(a=M, size=2000, replace=False)
plt.scatter(distances['original'][s], distances['proj_ortho'][s])
plt.axis('equal');

In [ ]:
s = rng.choice(a=M, size=10000, replace=False)

fig, ax = plt.subplots()

ratio_distributions = dict()

bins = np.linspace(0,1,100)
for key, d in distances.items():
    if key == 'original':
        continue
    elif key == 'random_filtered':
        continue
    ratio_distributions[key] = d/distances['original']
    ax.hist(ratio_distributions[key][s], bins=bins, alpha=0.25, label=key)
ax.legend()


In [ ]:
np.nanmin(ratio_distributions['rf_manual'])